In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc pandas qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Quantum Portfolio Optimizer : une Qiskit Function par Global Data Quantum


*Voir la [référence API](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer)*

> **Note:** Les Qiskit Functions sont une fonctionnalité expérimentale réservée aux utilisateurs des plans IBM Quantum&reg; Premium, Flex et On-Prem (via l'API IBM Quantum Platform). Elles sont en version préliminaire et peuvent évoluer.
## Vue d'ensemble
Le Quantum Portfolio Optimizer est une Qiskit Function qui s'attaque au problème d'optimisation dynamique de portefeuille, un problème classique en finance visant à rééquilibrer périodiquement des investissements sur un ensemble d'actifs afin de maximiser les rendements et de minimiser les risques. En déployant des techniques d'optimisation quantique de pointe, cette fonction simplifie le processus pour que les utilisateurs, sans expertise particulière en informatique quantique, puissent profiter de ses avantages dans la recherche de trajectoires d'investissement optimales. Idéal pour les gestionnaires de portefeuille, les chercheurs en finance quantitative et les investisseurs individuels, cet outil permet le backtesting de stratégies de trading dans le cadre de l'optimisation de portefeuille.
## Description de la fonction
Le Quantum Portfolio Optimizer utilise l'algorithme Variational Quantum Eigensolver (VQE) pour résoudre un problème d'optimisation binaire quadratique sans contrainte (QUBO), en traitant des problèmes d'optimisation dynamique de portefeuille. Il suffit de fournir les données de prix des actifs et de définir la contrainte d'investissement ; la fonction exécute ensuite le processus d'optimisation quantique et retourne un ensemble de trajectoires d'investissement optimisées.

Le processus comporte quatre étapes principales. D'abord, les données d'entrée sont transformées en un problème compatible quantique : on construit le QUBO du problème d'optimisation dynamique de portefeuille et on le convertit en un opérateur quantique (Hamiltonien d'Ising). Ensuite, le problème d'entrée et l'algorithme VQE sont adaptés pour s'exécuter sur le matériel quantique. Le VQE est alors exécuté sur ce matériel, et enfin, les résultats sont post-traités pour fournir les trajectoires d'investissement optimales. Le système intègre également un post-traitement tenant compte du bruit (basé sur [SQD](/guides/qiskit-addons-sqd)) afin de maximiser la qualité des résultats.

Cette Qiskit Function est basée sur le [manuscrit publié](https://arxiv.org/abs/2412.19150) par Global Data Quantum.
![Visualisation du flux de travail de la fonction](../docs/images/guides/global-data-quantum-optimizer/function_workflow.svg)
# Prise en main
Authentifie-toi avec ta [clé API](https://quantum.cloud.ibm.com/) et sélectionne la Qiskit Function comme suit. (Cet extrait suppose que tu as déjà [enregistré ton compte](/guides/functions#install-qiskit-functions-catalog-client) dans ton environnement local.)

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access function
dpo_solver = catalog.load("global-data-quantum/quantum-portfolio-optimizer")

## Exemple : optimisation dynamique de portefeuille avec sept actifs
Cet exemple montre comment exécuter la fonction d'optimisation dynamique de portefeuille (DPO) et ajuster ses paramètres pour des performances optimales. Il inclut des étapes détaillées pour affiner les paramètres et atteindre les résultats souhaités.

Ce cas implique sept actifs, quatre pas de temps et quatre qubits de résolution, pour un besoin total de 112 qubits.
### 1. Lire les actifs inclus dans le portefeuille
Si tous les actifs du portefeuille sont stockés dans un dossier à un chemin spécifique, tu peux les charger dans un `pandas.DataFrame` et le convertir en objet `dict` à l'aide de la fonction suivante.

In [ ]:
import os
import glob
import pandas as pd


def read_and_join_csv(file_pattern):
    """
    Reads multiple CSV files matching the file pattern and combines them
    into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all CSV files.
    """
    # Find all files matching the pattern
    csv_files = glob.glob(file_pattern)
    # Get the base file names without the .csv extension
    file_names = [os.path.basename(f).replace(".csv", "") for f in csv_files]
    # Read each CSV file into a DataFrame and set the first column as the index
    df_list = [pd.read_csv(f).set_index("Unnamed: 0") for f in csv_files]

    # Rename columns in each DataFrame to the base file names
    for df, name in zip(df_list, file_names):
        df.columns = [name]

    # Combine all DataFrames into one by merging them side by side
    combined_df = pd.concat(df_list, axis=1)
    return combined_df


file_pattern = "route/to/folder/with/assets/data/*.csv"
assets = read_and_join_csv(file_pattern).to_dict()

Pour cet exemple, nous avons utilisé les actifs [8801.T](https://finance.yahoo.com/quote/8801.T), [CLF](https://finance.yahoo.com/quote/CLF), [GBPJPY](https://finance.yahoo.com/quote/GBPJPY), [ITX.MC](https://finance.yahoo.com/quote/ITX.MC), [META](https://finance.yahoo.com/quote/META), [TMBMKDE-10Y](https://finance.yahoo.com/quote/TMBMKDE-10Y) et [XS2239553048](https://finance.yahoo.com/quote/XS2239553048). La figure suivante illustre les données utilisées dans cet exemple, montrant l'évolution du prix de clôture journalier des actifs du 1er janvier au 1er septembre 2023.

Dans cet exemple, pour garantir l'uniformité entre les dates, nous avons comblé les jours non ouvrés avec le prix de clôture de la dernière date disponible. Cette étape est nécessaire car les actifs sélectionnés proviennent de différents marchés avec des jours de bourse variables, et il est essentiel de standardiser le jeu de données pour assurer la cohérence.
![Visualisation des données historiques des actifs](../docs/images/guides/global-data-quantum-optimizer/assets.avif)
### 2. Définir le problème
Définis les spécifications du problème en configurant les paramètres du dictionnaire `qubo_settings`.

In [ ]:
qubo_settings = {
    "nt": 4,
    "nq": 4,
    "dt": 30,
    "max_investment": 25,
    "risk_aversion": 1000.0,
    "transaction_fee": 0.01,
    "restriction_coeff": 1.0,
}

### 3. Définir les paramètres de l'optimiseur et de l'ansatz (Optionnel)
Définis optionnellement des exigences spécifiques pour le processus d'optimisation, notamment le choix de l'optimiseur et de ses paramètres, ainsi que la spécification de la primitive et de ses configurations.

Pour l'ansatz Tailored, la taille de population choisie est basée sur des expériences précédentes montrant que cette valeur produit une optimisation stable et efficace.

Dans le cas de l'ansatz Real Amplitudes, tu peux suivre une relation linéaire entre la ``population_size`` et le nombre de qubits dans le circuit. À titre de règle empirique approximative, il est recommandé d'utiliser une ``population_size`` **minimale** ``~ 0.8 * n_qubits`` pour l'ansatz `real_amplitudes`.

On s'attend à ce que l'Optimized Real Amplitudes offre de meilleures performances d'optimisation que l'ansatz Real Amplitudes. Cependant, le nombre de variables à optimiser dans cet ansatz augmente bien plus vite que dans le cas Real Amplitudes (voir le [manuscrit](https://arxiv.org/pdf/2412.19150)). Ainsi, pour les grands problèmes, l'Optimized Real Amplitudes nécessite davantage d'exécutions de circuits. Il est susceptible d'être utile pour des problèmes jusqu'à 100 qubits, mais il est recommandé d'être attentif lors du réglage des paramètres ``population_size``. À titre d'exemple de cette mise à l'échelle de ``population_size``, le tableau précédent montre que pour un problème à 84 qubits, l'Optimized Real Amplitudes requiert une ``population_size`` de 120, tandis que pour un problème à 56 qubits, une ``population_size`` de 40 suffit.

In [ ]:
optimizer_settings = {
    "de_optimizer_settings": {
        "num_generations": 20,
        "population_size": 90,
        "recombination": 0.4,
        "max_parallel_jobs": 5,
        "max_batchsize": 4,
        "mutation_range": [0.0, 0.25],
    },
    "optimizer": "differential_evolution",
    "primitive_settings": {
        "estimator_shots": 25_000,
        "estimator_precision": None,
        "sampler_shots": 100_000,
    },
}

Il est également possible de choisir un ansatz spécifique. L'exemple suivant utilise l'ansatz ``'Tailored'``.

In [ ]:
ansatz_settings = {
    "ansatz": "tailored",
    "multiple_passmanager": False,
}

### 4. Exécuter le problème

In [ ]:
dpo_job = dpo_solver.run(
    assets=assets,
    qubo_settings=qubo_settings,
    optimizer_settings=optimizer_settings,
    ansatz_settings=ansatz_settings,
    backend_name="<backend name>",
    previous_session_id=[],
    apply_postprocess=True,
)

### 5. Récupérer les résultats
La fonction retourne un dictionnaire avec les trajectoires d'investissement triées du plus bas au plus élevé selon leur valeur de fonction objectif (voir la section [Sortie](https://docs.quantum.ibm.com/api/functions/global-data-quantum-optimizer#output) de la référence API). Cet ensemble de résultats permet d'identifier la trajectoire au coût le plus bas et ses évaluations d'investissement correspondantes. De plus, il permet d'analyser différentes trajectoires, facilitant la sélection de celles qui correspondent le mieux à des besoins ou objectifs spécifiques. Cette flexibilité garantit que les choix peuvent être adaptés à une grande variété de préférences ou de scénarios.
Commence par présenter la stratégie résultante ayant atteint le coût objectif le plus bas trouvé au cours du processus.

In [ ]:
# Get the results of the job
dpo_result = dpo_job.result()

# Show the solution strategy
dpo_result["result"]

{'time_step_0': {'8801.T': 0.11764705882352941,
  'ITX.MC': 0.20588235294117646,
  'META': 0.38235294117647056,
  'GBPJPY=X': 0.058823529411764705,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.058823529411764705,
  'XS2239553048': 0.17647058823529413},
 'time_step_1': {'8801.T': 0.11428571428571428,
  'ITX.MC': 0.14285714285714285,
  'META': 0.2,
  'GBPJPY=X': 0.02857142857142857,
  'TMBMKDE-10Y': 0.42857142857142855,
  'CLF': 0.0,
  'XS2239553048': 0.08571428571428572},
 'time_step_2': {'8801.T': 0.0,
  'ITX.MC': 0.09375,
  'META': 0.3125,
  'GBPJPY=X': 0.34375,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.25},
 'time_step_3': {'8801.T': 0.3939393939393939,
  'ITX.MC': 0.09090909090909091,
  'META': 0.12121212121212122,
  'GBPJPY=X': 0.18181818181818182,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.21212121212121213}}

Afterwards, using the metadata, you can access the results of all the sampled strategies. You can thereby further analyze the alternative trajectories returned by the optimizer. To do this, read the dictionary stored in `dpo_result['metadata']['all_samples_metrics']`, which contains not only additional information about the optimal strategy, but also details of the other candidate strategies evaluated during the optimization.

The following example shows how to read this information using `pandas` to extract key metrics associated with the optimal strategy. These include Restriction Deviation, Sharpe Ratio, and the corresponding investment return.

In [ ]:
# Convert metadata to a DataFrame
df = pd.DataFrame(dpo_result["metadata"]["all_samples_metrics"])

# Find the minimum objective cost
min_cost = df["objective_costs"].min()
print(f"Minimum Objective Cost Found: {min_cost:.2f}")

# Extract the row with the lowest cost
best_row = df[df["objective_costs"] == min_cost].iloc[0]

# Display the results associated with the best solution
print("Best Solution:")
print(f"  - Restriction Deviation: {best_row['rest_breaches']}%")
print(f"  - Sharpe Ratio: {best_row['sharpe_ratios']:.2f}")
print(f"  - Return: {best_row['returns']}")

Minimum Objective Cost Found: -3.78
Best Solution:
  - Restriction Deviation: 40.0
  - Sharpe Ratio: 24.82
  - Return: 0.46


Ensuite, grâce aux métadonnées, tu peux accéder aux résultats de toutes les stratégies échantillonnées. Tu peux ainsi analyser plus en détail les trajectoires alternatives retournées par l'optimiseur. Pour ce faire, lis le dictionnaire stocké dans `dpo_result['metadata']['all_samples_metrics']`, qui contient non seulement des informations supplémentaires sur la stratégie optimale, mais aussi des détails sur les autres stratégies candidates évaluées durant l'optimisation.

L'exemple suivant montre comment lire ces informations avec `pandas` pour extraire les métriques clés associées à la stratégie optimale. Celles-ci comprennent la déviation de contrainte, le ratio de Sharpe et le rendement d'investissement correspondant.